In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia
!pip install rank_bm25
!pip install pypdf
!pip install langchain_community
!pip install bitsandbytes
!pip install accelerate
!pip install weaviate-client
!pip install cohere
!pip install sentence_transformers
!pip install llama-cpp-python
!pip -q install google-generativeai
!pip install unstructured
!pip install flashrank
!pip install faiss-cpu
%%time
%pip install --upgrade --quiet  langchain-google-genai

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader, DirectoryLoader, WebBaseLoader
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter,CharacterTextSplitter, Language, SemanticChunker
from langchain_community.retrievers import WikipediaRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.weaviate_hybrid_search import WeaviateHybridSearchRetriever
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
)
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import ChatMessagePromptTemplate, PromptTemplate
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever
from langchain.document_transformers import LongContextReorder
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.llms import LlamaCpp
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from flashrank.Ranker import Ranker, RerankRequest
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain_core.prompts import ChatPromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings,HuggingFaceBgeEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain.prompts import PromptTemplate
from langchain import HuggingFacePipeline
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chains import RetrievalQA
from google.colab import userdata
from google.colab import drive
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain.load import dumps, loads
import os
import re
from re import search
import textwrap
import torch
import weaviate
import pandas as pd
import numpy as np

load_dotenv()
model = ChatOpenAI()

WEAVIATE_CLUSTER="https://hybridsearch-ewd5zpr1.weaviate.network"
WEAVIATE_API_KEY=""
WEAVIATE_URL = WEAVIATE_CLUSTER
WEAVIATE_API_KEY = WEAVIATE_API_KEY
HF_TOKEN=""

client = weaviate.Client(
    url=WEAVIATE_URL, auth_client_secret=weaviate.AuthApiKey(WEAVIATE_API_KEY),
    additional_headers={
         "X-HuggingFace-Api-Key": HF_TOKEN
    },
)
client.is_ready()
client.schema.get()
client.schema.delete_all()

In [ ]:
schema = {
    "classes": [
        {
            "class": "RAG",
            "description": "Documents for RAG",
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {"text2vec-huggingface": {"model": "sentence-transformers/all-MiniLM-L6-v2", "type": "text"}},
            "properties": [
                {
                    "dataType": ["text"],
                    "description": "The content of the paragraph",
                    "moduleConfig": {
                        "text2vec-huggingface": {
                            "skip": False,
                            "vectorizePropertyName": False,
                        }
                    },
                    "name": "content",
                },
            ],
        },
    ]
}

client.schema.create(schema)
client.schema.get()

In [ ]:
model_name = "HuggingFaceH4/zephyr-7b-beta"

def load_quantized_model(model_name: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

def initialize_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1
    return tokenizer

retriever = WeaviateHybridSearchRetriever(
  alpha = 0.5,
  client = client,
  index_name = "RAG"
  text_key = "content",
  attributes = [],
  create_schema_if_missing = True,
)

tokenizer = initialize_tokenizer(model_name)
model = load_quantized_model(model_name)

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    do_sample=True,
    top_k=5,
    max_new_tokens=100,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline)

In [ ]:
doc_path="/content/Retrieval-Augmented-Generation-for-NLP.pdf"
loader = PyPDFLoader(doc_path)
docs = loader.load()
docs, docs[6]

In [ ]:
retriever.add_documents(docs)
print(retriever.invoke("what is RAG token?")[0].page_content)
retriever.invoke(
    "what is RAG token?",
    score=True
)

In [ ]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}"),
    ]
)

template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.
When providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.
Always say "Do you have any more questions pertaining to this instrument?" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)
question_generator = create_stuff_documents_chain(llm, prompt)
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
)
result1 = hybrid_chain.invoke("what is natural language processing?")
print(result1)
print(result1['result'])

In [ ]:
query="What is Abstractive Question Answering?"
response = hybrid_chain.invoke({"query":query})

rag_chain =(
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)
query="what is RAG token?"
response=rag_chain.invoke("what is RAG token?")
print(response)
print(response["result"])

In [ ]:
compressor = CohereRerank()
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

compression_docs = compression_retriever.get_relevant_documents(query)
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
)

response = hybrid_chain.invoke({"query":query})
response = hybrid_chain.invoke("What is Abstractive Question Answering?")
print(response.get("result"))

### **Re-Rank**

In [ ]:
documents = [
    "This is a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings.",
    "Understanding document structure aids in keyword extraction.",
    "Efficient keyword extraction enhances search accuracy.",
    "Semantic similarity improves document retrieval performance.",
    "Machine learning algorithms can optimize keyword extraction methods."
]

model_name = 'sentence-transformers/paraphrase-xlm-r-multilingual-v1'
model = SentenceTransformer(model_name)
document_embeddings = model.encode(documents)

len(documents)
len(document_embeddings)
len(document_embeddings[0])

for i, embedding in enumerate(document_embeddings):
  print(f"Document {i+1} embedding: {embedding}")

In [ ]:
query = "Natural language processing techniques enhance keyword extraction efficiency."
query_embedding = model.encode(query)
print("Query embedding:", query_embedding)

In [ ]:
similarities = cosine_similarity(np.array([query_embedding]), document_embeddings)
similarities

In [ ]:
most_similar_index = np.argmax(similarities)
most_similar_index

In [ ]:
most_similar_document = documents[most_similar_index]
most_similar_document

In [ ]:
similarity_score = similarities[0][most_similar_index]
similarity_score

In [ ]:
sorted_indices = np.argsort(similarities[0])[::-1]
sorted_indices

In [ ]:
ranked_documents = [(documents[i], similarities[0][i]) for i in sorted_indices]
ranked_documents

In [ ]:
print("Ranked Documents:")
for rank, (document, similarity) in enumerate(ranked_documents, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

print("Top 4 Documents:")
for rank, (document, similarity) in enumerate(ranked_documents[:4], start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

In [ ]:
top_4_documents = [doc[0] for doc in ranked_documents[:4]]
top_4_documents

In [ ]:
tokenized_top_4_documents = [doc.split() for doc in top_4_documents]
tokenized_top_4_documents

In [ ]:
tokenized_query = query.split()
tokenized_query

In [ ]:
bm25=BM25Okapi(tokenized_top_4_documents)
bm25_scores=bm25.get_scores(tokenized_query)
bm25, bm25_scores

In [ ]:
sorted_indices2 = np.argsort(bm25_scores)[::-1]
reranked_documents = [(top_4_documents[i], bm25_scores[i]) for i in sorted_indices2]

print(sorted_indices2)
print(reranked_documents)
print("Rerank of top 4 Documents:")
for rank, (document, similarity) in enumerate(reranked_documents, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

ranked_documents[:4]

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
pairs = []
for doc in top_4_documents:
  pairs.append([query, doc])

In [ ]:
scores = cross_encoder.predict(pairs)
scored_docs = zip(scores, top_4_documents)
reranked_document_cross_encoder = sorted(scored_docs, reverse=True)

scores, scored_docs, reranked_document_cross_encoder

In [ ]:
from getpass import getpass
co = cohere.Client(getpass("Cohere API key: "))
response = co.rerank(
    model="rerank-english-v3.0",
    query="Natural language processing techniques enhance keyword extraction efficiency.",
    documents=top_4_documents,
    return_documents=True
)

print(response)
response.results[0].document.text
response.results[0].relevance_score

for i in range(4):
  print(f'text: {response.results[i].document.text} score: {response.results[i].relevance_score}')

**Merger Retriever and LongContextReorder**

In [ ]:
loader_harrypotter  = PyPDFLoader("/content/harry_potter_book.pdf")
documnet_harrypotter = loader_harrypotter.load()
print(len(documnet_harrypotter))


loader_got = PyPDFLoader("/content/got_book.pdf")
documnet_got = loader_got.load()
print(len(documnet_got))


text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)
text_got = text_splitter.split_documents(documnet_got)
print(len(text_harrypotter))
print(len(text_got))

In [ ]:
HF_TOKEN_REMOVEDembeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
HF_TOKEN_REMOVEDbge_embeddings = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-large-en")
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY
openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

In [ ]:
os.getcwd()
CURRENT_DIR = os.path.dirname(os.path.abspath("."))
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")

DB_DIR, CURRENT_DIR

In [ ]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False
)

harrypotter_vectorstore = Chroma.from_documents(
    text_harrypotter,
    HF_TOKEN_REMOVEDembeddings,
    collection_name="harrypotter",
    collection_metadata={"hnsw": "cosine"},
    persist_directory="/store/harrypotter"
)


got_vectorstore = Chroma.from_documents(
    text_got,
    HF_TOKEN_REMOVEDbge_embeddings,
    client_settings=client_settings,
    collection_name="got",
    collection_metadata={"hnsw":"cosine"},
    persist_directory="/store/got"
)

retriever_harrypotter = harrypotter_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5},
    "include_metadata": True
)

retriever_got = got_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5},
    "include_metadata": True
)

lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

for chunks in lotr.get_relevant_documents("Who was the jon snow?"):
  print(chunks.page_content)

for chunks in lotr.get_relevant_documents("Who is a harry potter?"):
  print(chunks.page_content)

In [ ]:
filter = EmbeddingsRedundantFilter(embeddings=HF_TOKEN_REMOVEDbge_embeddings)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(
    transformers=[filter, reordering]
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=lotr,
    search_kwargs={"k": 3, "include_metadata": True}
)

llms = LlamaCpp(
    streaming=True,
    model_path="/content/drive/MyDrive/zephyr-7b-beta.Q4_K_M.gguf",
    max_tokens=1500,
    temperature=0.75,
    top_p=1,
    gpu_layers=0,
    stream=True,
    verbose=True,
    n_threads = int(os.cpu_count() / 2),
    n_ctx=4096
)

qa = RetrievalQA.from_chain_type(
    llm=llms,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
)

query ="who is jon snow?"
results = qa(query)
print(results['result'])
print(results["source_documents"])

In [ ]:
results = qa("who is a harry potter?")
print(results['result'])
print(results["source_documents"])
for source in  results["source_documents"]:
    print(source.metadata)

In [ ]:
results = qa.invoke("How does Jon Snow's relationship with the Stark family influence his identity and decisions throughout A Game of Thrones?")

print(results['result'])
print(results["source_documents"])
print(results)
for source in  results["source_documents"]:
  print(source.metadata)

### **Rank Fusion**

In [ ]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

def wrap_text(text, width=90):
  lines = text.split('\n')
  wrapped_lines = [textwrap.fill(line, width=width) for line in lines]
  wrapped_text = '\n'.join(wrapped_lines)
  return wrapped_text

llm = ChatGoogleGenerativeAI(model="Write a ballad about LangChain")
result = llm.invoke("Write a ballad about LangChain")
print(result.content)

In [ ]:
drive.mount('/content/drive')
data_path="/content/drive/MyDrive/English"

loader = DirectoryLoader(data_path, glob="*.txt", show_progress=True)
docs = loader.load()
raw_text = ''
for i, doc in enumerate(docs):
  text = docs.page_content
  if text:
    raw_text += text

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.split_text(raw_text)

print(len(docs))
docs = docs[:50]
print(len(docs))
print(docs[0])
print(docs[2].page_content)
print(docs[1].page_content)
print(raw_text)
print(len(texts))
print(texts[0])

In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True}
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

db = Chroma.from_texts(
    texts,
    embedding_function,
    persist_directory="./chroma_db",
)

query = "Tell me about Universal Studios Singapore?"
retriever = db.as_retriever()
retriever.get_relevant_documents(query)
db.similarity_search(query, k=5)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

text_reply = chain.invoke("Tell me about Universal Studio Singapore")
print(wrap_text(text_reply))

In [ ]:
import langchain

prompt = ChatPromptTemplate(
    input_variables=['question'],
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                template='You are a helpful assistant that generates multiple search queries based on a single input query.'
            )
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['question'],
                template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'
              )
        )
    ]
)

original_query = "universal studios Singapore"
generate_queries = (
    prompt | llm | StrOutputParser() | (lambda x: x.split('\n'))
)

def reciprocal_rank_fusion(results: list[list], k=60):
  fused_scores = {}
  for docs in results:
    for rank, doc in enumerate(docs):
      doc_str = dumps(doc)
      if doc_str not in fused_scores:
        fused_scores[doc_str] = 0
      else:
        fused_scores[doc_str] += 1 / (rank + k)

  reranked_results = [
      (loads(doc_str), score)
      for doc_str, score in sorted(
          fused_scores.items(),
          key=lambda x: x[1],
          reverse=True
      )
  ]

  return reranked_results

ragfusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion
langchain.debug = True
ragfusion_chain.input_schema.schema()
ragfusion_chain.invoke({"question": original_query})

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
full_rag_fusion_chain = (
    {"context": ragfusion_chain, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

full_rag_fusion_chain.input_schema.schema()
full_rag_fusion_chain.invoke({"question": "Tell me about Singapore’s nightlife scene?"})

### **Flash Rerank**

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

query = "How to speedup LLMs?"

passages = [
   {
      "id":1,
      "text":"Introduce *lookahead decoding*: - a parallel decoding algo to accelerate LLM inference - w/o the need for a draft model or a data store - linearly decreases # decoding steps relative to log(FLOPs) used per decoding step.",
      "meta": {"additional": "info1"}
   },
   {
      "id":2,
      "text":"LLM inference efficiency will be one of the most crucial topics for both industry and academia, simply because the more efficient you are, the more $$$ you will save. vllm project is a must-read for this direction, and now they have just released the paper",
      "meta": {"additional": "info2"}
   },
   {
      "id":3,
      "text":"There are many ways to increase LLM inference throughput (tokens/second) and decrease memory footprint, sometimes at the same time. Here are a few methods I’ve found effective when working with Llama 2. These methods are all well-integrated with Hugging Face. This list is far from exhaustive; some of these techniques can be used in combination with each other and there are plenty of others to try. - Bettertransformer (Optimum Library): Simply call `model.to_bettertransformer()` on your Hugging Face model for a modest improvement in tokens per second. - Fp4 Mixed-Precision (Bitsandbytes): Requires minimal configuration and dramatically reduces the model's memory footprint. - AutoGPTQ: Time-consuming but leads to a much smaller model and faster inference. The quantization is a one-time cost that pays off in the long run.",
      "meta": {"additional": "info3"}

   },
   {
      "id":4,
      "text":"Ever want to make your LLM inference go brrrrr but got stuck at implementing speculative decoding and finding the suitable draft model? No more pain! Thrilled to unveil Medusa, a simple framework that removes the annoying draft model while getting 2x speedup.",
      "meta": {"additional": "info4"}
   },
   {
      "id":5,
      "text":"vLLM is a fast and easy-to-use library for LLM inference and serving. vLLM is fast with: State-of-the-art serving throughput Efficient management of attention key and value memory with PagedAttention Continuous batching of incoming requests Optimized CUDA kernels",
      "meta": {"additional": "info5"}
   }
]

def get_result(query,passages,choice):
  if choice == "Nano":
    ranker = Ranker()
  elif choice == "Small":
    ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/opt")
  elif choice == "Medium":
    ranker = Ranker(model_name="rank-T5-flan", cache_dir="/opt")
  elif choice == "Large":
    ranker = Ranker(model_name="ms-marco-MultiBERT-L-12", cache_dir="/opt")

  rerankrequest = RerankRequest(query=query, documents=passages)
  results = ranker.rerank(rerankrequest)
  print(results)

  return results


%%time
print("Emon")
%%time
get_result(query,passages,"Nano")
%%time
get_result(query,passages,"Small")
%%time
get_result(query,passages,"Medium")

In [ ]:
documents = TextLoader("/content/state_of_the_union.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

for id, text in enumerate(texts):
    text.metadata["id"] = id

In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-ada-002")
retriever = FAISS.from_documents(texts, embedding).as_retriever(search_kwargs={"k": 10})


query = "What did the president say about Ketanji Brown Jackson"
docs = retriever.invoke(query)
pretty_print_docs(docs)

In [ ]:
llm = ChatOpenAI(temperature=0)
compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

compression_docs = compression_retriever.get_relevant_documents("What did the president say about Ketanji Jackson Brown")
pretty_print_docs(compression_docs)
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

print(len(compression_docs))
print(compression_docs)
print([doc.metadata["id"] for doc in compression_docs])
pretty_print_docs(compression_docs)
chain.invoke(query)